# MindTrack: Depression Prediction Walkthrough

This notebook takes a reviewer through the analysis from the source survey files to the depression-model evaluation. It is organised in the order the work was done: inspect the raw data, conduct EDA, prepare the analysis table, explore predictor structure with PCA, split the data, train and compare models, then interpret the results.

**There is one prediction target:** the project-defined Beck score group of 11 or higher. The separate phone-dependence prediction task has been dropped. The recalculated phone-dependence total remains a candidate *input* to the depression models, following the current analysis plan. It is not a second target.

The notebook is designed to be presented from top to bottom. Each section includes a short explanation followed by the relevant chart, table or code output.


### Before running the notebook

Open the project folder in VS Code and select the MindTrack Python environment as the notebook kernel. In the WSL terminal, from the project root, activate the environment and install dependencies if needed:

```bash
source ~/.venvs/mindtrack/bin/activate
python -m pip install -r analysis/eda/requirements-eda.txt
python -m pip install -r analysis/modeling/requirements-modeling.txt
```

Then run the notebook cells from top to bottom. The preparation, PCA and modelling cells call the project's scripts. The modelling cell reruns the comparison and refreshes files under `results/` and `models/`.


In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import confusion_matrix

ROOT = Path.cwd()
if not (ROOT / 'analysis').exists():
    ROOT = ROOT.parent
assert (ROOT / 'analysis').exists(), 'Open/run this notebook from the project folder or its notebooks folder.'
print('Project folder:', ROOT)


## 1. Study question and scope

**Study question:** How well can selected background and phone-use information estimate whether a participant's Beck total is below 11 or at least 11?

- **Only target:** `Higher_Beck_Score_11plus` (0 = below 11; 1 = 11 or above), based on the chosen project cutoff and score bands in the data dictionary.
- **Phone dependence:** the separate dependence-classification target is out of scope. The recalculated TMD total is still an input candidate in the current depression model.
- Beck item answers and the Beck total are excluded from the model inputs because they define the target.
- The 11+ group is a questionnaire score grouping, **not a clinical diagnosis**.
- The data are observational and cross-sectional. Associations and predictions do not show that phone use causes depressive symptoms.


## 2. Source files and record alignment

The source folder contains three workbooks used in this analysis: sociodemographic/mobile-use responses, phone-dependence responses, and depression responses. The code below checks that student codes are unique and that the same coded records appear in the same order in all three files. It prints structural counts only, not individual response rows.


In [ ]:
raw = ROOT / 'Digital Sedentary Behavior, Mobile Device Dependen'
sources = [
    ('Sociodemographic and mobile use', raw / '2. Dataset_Sociodemo.xlsx', 'SocioDemo_data'),
    ('Phone dependence', raw / '3. Dataset_MobileDependence.xlsx', 'TMD_data'),
    ('Beck scores', raw / '5. Dataset_Depression.xlsx', 'Depression_data'),
]
frames = {label: pd.read_excel(path, sheet_name=sheet) for label, path, sheet in sources}
inventory = pd.DataFrame([{'source': label, 'records': len(frame), 'columns': len(frame.columns), 'unique_codes': frame['Code'].nunique(), 'missing_codes': int(frame['Code'].isna().sum())} for label, frame in frames.items()])
display(inventory)
codes = [frame['Code'] for frame in frames.values()]
assert all(code.is_unique for code in codes), 'Duplicate/missing student codes need investigation.'
assert codes[0].equals(codes[1]) and codes[0].equals(codes[2]), 'Student records do not align across workbooks.'
print(f'All {len(codes[0]):,} records align across the three source files.')

## 3. Exploratory data analysis (EDA)

EDA describes the depression scores and candidate predictors **before data preparation and modelling**. The charts are saved in `analysis/eda/figures/presentation/`; supporting findings and quality checks are in `analysis/eda/preliminary_eda_findings.md` and `analysis/eda/tables/`.

**Current descriptive picture:** 1,220 aligned records; 218 (17.9%) are in the project-defined 11+ score group. Beck scores cluster at the low end. The recalculated phone-dependence total has a weak positive rank association with Beck total. One supplied dependence total disagreed with its 22 responses, and only one student selected the 1–3 hour daily-use category. These are descriptive patterns, not evidence of cause or diagnosis.


### The outcome: Beck score distribution and group definition

These charts show the score distribution and the score bands used to describe the data. For this project, scores below 11 form one class and scores of 11 or higher form the other. This combines the 11–16 mild mood-disturbance band with all higher bands; it does not mean that everyone in the 11+ class has a clinical diagnosis. Verify that the questionnaire version matches the scoring guide. The grouping creates an imbalanced target, so accuracy alone will not tell us whether a model identifies the higher-score group.

![Distribution of Beck total scores](../analysis/eda/figures/presentation/01_beck_score_distribution.png)

![Beck score bands from the project codebook](../analysis/eda/figures/presentation/02_beck_score_bands.png)


### Phone-use candidate predictors (not another prediction target)

These charts compare phone-use responses and the recalculated phone-dependence score with Beck scores. Phone dependence is being considered as an input for predicting the depression grouping; the project is **not** training a separate dependence model. Small categories, especially the daily-use 1–3 hour group and the phone-argument “Always” group, should be interpreted cautiously.

![Recalculated phone-dependence score and Beck total](../analysis/eda/figures/presentation/03_phone_dependence_vs_beck.png)

![Beck score by phone-dependence band](../analysis/eda/figures/presentation/04_beck_score_by_dependence_band.png)

![Higher-score-group rate by phone-related arguments](../analysis/eda/figures/presentation/05_beck_flag_by_phone_arguments.png)

![Daily phone-use category distribution](../analysis/eda/figures/presentation/06_average_daily_use_distribution.png)


### Other candidate predictor patterns

These charts show age, differences in observed higher-score-group rates across departments and response patterns across selected phone/family items. The final model only uses variables permitted by the modelling plan; a visible association in a chart does not by itself make a variable useful or appropriate for prediction.

![Age distribution](../analysis/eda/figures/presentation/07_age_distribution.png)

![Higher-score-group rate by department](../analysis/eda/figures/presentation/08_beck_flag_by_department.png)

![Higher-score-group rate across ordinal phone and family responses](../analysis/eda/figures/presentation/09_ordinal_predictor_patterns.png)

![Response patterns across the 22 phone-dependence items](../analysis/eda/figures/presentation/10_tmd_item_response_heatmap.png)


## 4. Cleaning and analysis-table preparation

The preparation script checks ID alignment, missingness and score consistency. The current audit found no duplicate IDs/full rows or missing values in the selected fields, so no student records were removed. It recalculates the phone-dependence total because one supplied total did not match the sum of its 22 answers. It keeps the depression score for outcome construction and description, but excludes Beck responses and Beck-derived values from model inputs to prevent target leakage. The original workbooks are not edited.

The next cell reruns the preparation script and displays the resulting analysis table summary.


In [ ]:
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'cleaning' / 'prepare_depression_dataset.py')], check=True)
prepared = pd.read_csv(ROOT / 'data' / 'processed' / 'depression_analysis_dataset.csv')
print('Prepared table shape:', prepared.shape)
print('Missing cells in prepared table:', int(prepared.isna().sum().sum()))
display(prepared['Higher_Beck_Score_11plus'].value_counts().rename(index={0: 'Below 11', 1: '11+ score group'}).to_frame('students'))
display(Markdown((ROOT / 'data' / 'processed' / 'preparation_log.md').read_text()))

## 5. PCA: explore the structure of predictors

PCA compresses correlated input variables into components that summarise predictor variation. It is **not** a supervised method and does not identify which feature has the greatest effect on depression. For this descriptive PCA, nominal variables are one-hot encoded, ordered responses retain their order, inputs are standardised, and Beck score, target and student ID are excluded from the PCA calculation. The target is used only to colour the PC1–PC2 visualisation.

The current exploratory PCA has 57 encoded dimensions. PC1 and PC2 explain 20.8% of predictor variation together; 27 components are needed to retain 80%. The two score groups overlap in the PC1–PC2 view. This PCA is descriptive and fitted on the full dataset for exploration. In the predictive comparison, any PCA is fitted within the training folds to avoid letting validation or test data influence the transformation.


![PCA scree and cumulative variance](../analysis/pca/figures/01_pca_scree_cumulative_variance.png)

![Students projected onto the first two principal components](../analysis/pca/figures/02_pca_pc1_pc2_by_beck_group.png)

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'pca' / 'run_pca_eda.py')], check=True)
display(pd.read_csv(ROOT / 'analysis' / 'pca' / 'tables' / 'pca_explained_variance.csv').head(10).round(3))
display(pd.read_csv(ROOT / 'analysis' / 'pca' / 'tables' / 'pca_loadings_top10.csv').assign(max_abs_loading=lambda x: x.iloc[:, 1:].abs().max(axis=1)).sort_values('max_abs_loading', ascending=False).head(15).round(3))

## 6. Split, train and compare depression models

The split is made **after the fixed data checks and target construction**. The 80% portion is the development/training set, not a “cleaning set”; it is used for repeated cross-validation and model fitting. The held-out 20% is used once for final evaluation. With the current data and seed, the split contains 976 training records (174 in the 11+ group) and 244 test records (44 in the 11+ group). The script saves the exact split as `data/splits/depression_train_80pct.csv` and `data/splits/depression_test_20pct.csv`. To create those files without fitting models, run `python analysis/modeling/run_depression_models.py --split-only`.

The comparison includes a most-common-class baseline and four classifiers: logistic regression, decision tree, random forest and gradient boosting. Each classifier is evaluated with and without PCA. Repeated stratified five-fold cross-validation is repeated three times on the training set. Encoding, scaling and PCA are fitted inside each fold. The model is selected using mean cross-validated PR-AUC. Test metrics use a fixed probability threshold of 0.50; this is a modelling choice, not a diagnostic threshold.


### Run the modelling comparison

The next cell reruns the analysis script. It may take a little while because each classifier is evaluated across repeated stratified cross-validation folds. When it finishes, it saves the comparison table, predictions, permutation importance and fitted best model in the project folders.


In [ ]:
# Run the full comparison, save outputs, and show a compact table.
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'modeling' / 'run_depression_models.py')], check=True)
model_results = pd.read_csv(ROOT / 'results' / 'depression_model_comparison.csv')

comparison = model_results[[
    'model', 'pca', 'cv_pr_auc_mean', 'test_pr_auc',
    'test_precision', 'test_recall'
]].sort_values('cv_pr_auc_mean', ascending=False)
comparison = comparison.rename(columns={
    'model': 'Model', 'pca': 'PCA', 'cv_pr_auc_mean': 'CV PR-AUC',
    'test_pr_auc': 'Test PR-AUC', 'test_precision': 'Test precision',
    'test_recall': 'Test recall'
})
display(comparison.round(3))


In [ ]:
# Read the CV-selected model's held-out metrics and its test-set confusion matrix.
candidates = model_results[model_results['model'] != 'Baseline (most common)']
best_row = candidates.loc[candidates['cv_pr_auc_mean'].idxmax()]
print(f"Selected using training cross-validation PR-AUC: {best_row['model']} (PCA: {best_row['pca']})")

metric_columns = [
    'test_accuracy', 'test_balanced_accuracy', 'test_precision', 'test_recall',
    'test_f1', 'test_roc_auc', 'test_pr_auc'
]
metric_labels = [
    'Accuracy', 'Balanced accuracy', 'Precision', 'Recall', 'F1-score',
    'ROC-AUC', 'PR-AUC'
]
selected_metrics = best_row[metric_columns].astype(float).to_frame('Test result')
selected_metrics.index = metric_labels
display(selected_metrics.round(3))

predictions = pd.read_csv(ROOT / 'results' / 'best_model_test_predictions.csv')
confusion = confusion_matrix(
    predictions['actual_higher_beck_score_11plus'],
    predictions['predicted_higher_beck_score_11plus'],
    labels=[0, 1]
)
confusion_table = pd.DataFrame(
    confusion,
    index=['Actual below 11', 'Actual 11+'],
    columns=['Predicted below 11', 'Predicted 11+']
)
print('Held-out test confusion matrix:')
display(confusion_table)


### Which inputs help prediction?

Permutation importance estimates how much the selected model's PR-AUC drops when one original input is shuffled. A larger drop suggests the model relied more on that input for predictions in this dataset. It does **not** prove that the input causes depressive symptoms. The importance is calculated on the held-out test data for description; it is not used to choose the model or select features. PCA loadings answer a different question and are not a substitute for this target-focused check.


In [ ]:
importance = pd.read_csv(ROOT / 'results' / 'best_model_permutation_importance.csv')
display(importance.head(10).round(4))
display(Markdown((ROOT / 'results' / 'modeling_summary.md').read_text()))


## 7. What to say about the results

Use the tables above to describe the model comparison. The model named “selected” is selected by **cross-validation PR-AUC on training data**. The test results are reported after that selection; do not choose a model just because it happened to score highest on this one test split.

The current run selected gradient boosting without PCA by mean cross-validation PR-AUC (about 0.29). Its held-out test PR-AUC is about 0.27, compared with about 0.18 for the majority-class baseline. Its test precision is about 0.28 and recall about 0.55 at the fixed 0.50 threshold. These are modest results, so describe the model as showing limited predictive signal in this dataset, not as a reliable depression screener. Random forest without PCA had a higher PR-AUC on this particular test split, but the test split must not be used to select a model.

PCA was tested as an alternative input transformation, but the CV-selected model did not use PCA. PCA helped describe predictor structure; it did not establish which features matter for depression. The recalculated phone-dependence score remains a candidate input, while dependence as a separate prediction target has been dropped.

**Limits to mention:** the 11+ grouping combines several score bands and is not a diagnosis; confirm that the questionnaire version matches the scoring guide; the data are cross-sectional and from one university; performance is uncertain because the positive group is relatively small; and external validation is needed before considering wider use.

The notebook and scripts support a reproducible analysis walkthrough. Model export to Kotlin, agreement testing between Python and an app, external validation and deployment are separate future tasks.


### Next: run the app-ready model comparison

The app-ready workflow compares all four classifiers on the 27 candidate fields the form can collect and on a reduced field set selected inside each training fold. It keeps the TMD total as an input and omits PCA from this deployment-focused comparison. The holdout has already been reviewed, so treat its metrics as preliminary. See [the app input contract](../analysis/modeling/app_input_schema.md).


## 8. App-ready model and input selection

Run the next cell to compare the baseline and four classifiers on all app-compatible fields and the L1-selected field set. Training CV PR-AUC chooses the model and input set. Only that winner is evaluated on the saved holdout.

The selected field list is what the app needs to collect, plus the 22 TMD answers used to calculate the required dependence total. The result is an estimate of a Beck score group, not a diagnosis.


In [ ]:
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'modeling' / 'run_app_ready_models.py')], check=True)
app_results = pd.read_csv(ROOT / 'results' / 'app_ready_model_comparison.csv')
display(app_results[['model', 'input_set', 'cv_pr_auc_mean', 'cv_pr_auc_sd', 'selected_for_test', 'selected_compact_for_test', 'test_pr_auc', 'test_precision', 'test_recall']].sort_values('cv_pr_auc_mean', ascending=False).round(3))
selected_fields = pd.read_csv(ROOT / 'results' / 'app_ready_selected_fields.csv')
display(selected_fields[selected_fields['selected_by_l1']][['field', 'selection_score_abs_l1_coefficient_sum', 'always_keep']])model_inputs = pd.read_csv(ROOT / 'results' / 'app_ready_compact_model_inputs.csv')
display(model_inputs[model_inputs['required_by_selected_model']][['field', 'model', 'input_set']])compact_metrics = app_results[app_results['selected_compact_for_test']]
display(compact_metrics[['model', 'test_accuracy', 'test_precision', 'test_recall', 'test_pr_auc']].round(3))